# 11 Python intermediate - Abstrakcyjne klasy bazowe (`abc`)  v2.0

## Rozkład jazdy

1. 🔹 **Klasy abstrakcyjne i `abc.ABC`** - po co istnieją, jak działają
2. 🔹 **`@abstractmethod` i `@abstractproperty`** - wymuszanie implementacji
3. 🔹 **`isinstance()` / `issubclass()` z ABC** - wirtualne podklasy
   🐍 Ćwiczenia do każdej sekcji

---

## 1. 🔹 Klasy abstrakcyjne i `abc.ABC`

Klasa abstrakcyjna (abstract class) to klasa, której nie można
bezpośrednio instancjonować - służy wyłącznie jako szablon dla
podklas. W Pythonie klasy abstrakcyjne tworzymy dziedzicząc po
`abc.ABC` lub ustawiając `metaclass=abc.ABCMeta`.

Dzięki klasom abstrakcyjnym możemy:
- zdefiniować interfejs, który muszą implementować podklasy,
- uniknąć tworzenia obiektów niekompletnych klas,
- dokumentować kontrakt (contract) w czytelny sposób.

> 💡 `abc.ABC` to skrót od `ABCMeta`. Wystarczy po niej dziedziczyć
> - nie trzeba jawnie podawać `metaclass=ABCMeta`.

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    """Abstrakcyjna klasa bazowa dla figur geometrycznych."""

    @abstractmethod
    def area(self) -> float:
        """Zwraca pole figury."""

    @abstractmethod
    def perimeter(self) -> float:
        """Zwraca obwód figury."""

    def describe(self) -> str:
        return f"area={self.area():.2f}, perimeter={self.perimeter():.2f}"


# Shape()  # TypeError - nie można instancjonować klasy abstrakcyjnej


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        import math
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        import math
        return 2 * math.pi * self.radius


c = Circle(5)
print(c.describe())

> 💡 Jeśli podklasa nie zaimplementuje **wszystkich** metod
> abstrakcyjnych, sama stanie się klasą abstrakcyjną i też nie
> będzie można jej instancjonować.

---

### 🐍 Ćwiczenia - klasy abstrakcyjne

**Ćw. 1.** Utwórz abstrakcyjną klasę `Animal` z metodami
abstrakcyjnymi `speak()` i `move()`. Następnie utwórz podklasy
`Dog` i `Bird` implementujące obie metody. Wypisz wynik dla
obu zwierząt.

**Ćw. 2.** Utwórz abstrakcyjną klasę `Vehicle` z metodami
`start()`, `stop()` i właściwością `max_speed`. Zaimplementuj
podklasy `Car` i `Bicycle`.

**Ćw. 3.** *(Trudniejsze)* Utwórz abstrakcyjną klasę `Serializable`
z metodami `to_dict()` i `from_dict(cls, data)` (classmethod).
Zaimplementuj ją w klasie `Person(name, age)`.
# hint: from_dict powinno być jednocześnie @classmethod i @abstractmethod

In [ ]:
# Ćw. 1
from abc import ABC, abstractmethod

class Animal(ABC):
    ...


class Dog(Animal):
    ...


class Bird(Animal):
    ...


dog = Dog()
bird = Bird()
print(dog.speak(), dog.move())
print(bird.speak(), bird.move())

In [ ]:
# Ćw. 2
from abc import ABC, abstractmethod

class Vehicle(ABC):
    ...


class Car(Vehicle):
    ...


class Bicycle(Vehicle):
    ...


car = Car()
bike = Bicycle()
print(car.max_speed, car.start())
print(bike.max_speed, bike.start())

In [ ]:
# Ćw. 3
from abc import ABC, abstractmethod

class Serializable(ABC):
    @abstractmethod
    def to_dict(self) -> dict:
        ...

    @classmethod
    @abstractmethod
    def from_dict(cls, data: dict):
        ...


class Person(Serializable):
    def __init__(self, name: str, age: int):
        ...


p = Person('Alice', 30)
data = p.to_dict()
print(data)
p2 = Person.from_dict(data)
print(p2.name, p2.age)

---

## 2. 🔹 `@abstractmethod` i `@abstractproperty`

Dekorator `@abstractmethod` wymusza implementację metody w każdej
nieabstrakcyjnej podklasie. Można go łączyć z innymi dekoratorami:

| Kombinacja | Opis |
|---|---|
| `@abstractmethod` | zwykła metoda abstrakcyjna |
| `@property` + `@abstractmethod` | abstrakcyjna właściwość |
| `@classmethod` + `@abstractmethod` | abstrakcyjna metoda klasowa |
| `@staticmethod` + `@abstractmethod` | abstrakcyjna metoda statyczna |

> 💡 Kolejność dekoratorów ma znaczenie: `@abstractmethod` zawsze
> umieszczamy jako **wewnętrzny** (bliżej `def`).

In [ ]:
from abc import ABC, abstractmethod


class DataStore(ABC):

    @property
    @abstractmethod
    def name(self) -> str:
        """Nazwa magazynu danych."""

    @abstractmethod
    def save(self, key: str, value) -> None:
        """Zapisz wartość pod podanym kluczem."""

    @abstractmethod
    def load(self, key: str):
        """Wczytaj wartość spod klucza."""

    @classmethod
    @abstractmethod
    def from_config(cls, config: dict):
        """Utwórz instancję z konfiguracji."""


class InMemoryStore(DataStore):
    def __init__(self, store_name: str):
        self._name = store_name
        self._data: dict = {}

    @property
    def name(self) -> str:
        return self._name

    def save(self, key: str, value) -> None:
        self._data[key] = value

    def load(self, key: str):
        return self._data.get(key)

    @classmethod
    def from_config(cls, config: dict):
        return cls(config['name'])


store = InMemoryStore.from_config({'name': 'cache'})
store.save('user', {'id': 1, 'login': 'alice'})
print(store.name, store.load('user'))

---

### 🐍 Ćwiczenia - @abstractmethod

**Ćw. 4.** Utwórz abstrakcyjną klasę `Logger` z abstrakcyjną
właściwością `level` (str) oraz metodą `log(message: str)`.
Zaimplementuj `FileLogger` (zapisuje do pliku) i `ConsoleLogger`
(wypisuje na ekran).

**Ćw. 5.** Utwórz abstrakcyjną klasę `Parser` z metodami
`parse(text: str) -> dict` i `format(data: dict) -> str`.
Dodaj abstrakcyjną metodę klasową `supported_format() -> str`.
Zaimplementuj `JsonParser`.

**Ćw. 6.** *(Trudniejsze)* Utwórz abstrakcyjną klasę `Validator`
z metodą `validate(value) -> bool` i `error_message() -> str`.
Dodaj metodę `check(value)`, która wywołuje `validate()` i jeśli
zwróci False - rzuca `ValueError` z komunikatem z `error_message()`.
Zaimplementuj `RangeValidator(min_val, max_val)` i `LengthValidator
(min_len, max_len)`.

In [ ]:
# Ćw. 4
from abc import ABC, abstractmethod

class Logger(ABC):
    @property
    @abstractmethod
    def level(self) -> str:
        ...

    @abstractmethod
    def log(self, message: str) -> None:
        ...


class ConsoleLogger(Logger):
    ...


class FileLogger(Logger):
    def __init__(self, filepath: str):
        ...


logger = ConsoleLogger()
print(logger.level)
logger.log('Hello!')

In [ ]:
# Ćw. 5
from abc import ABC, abstractmethod
import json

class Parser(ABC):
    @abstractmethod
    def parse(self, text: str) -> dict:
        ...

    @abstractmethod
    def format(self, data: dict) -> str:
        ...

    @classmethod
    @abstractmethod
    def supported_format(cls) -> str:
        ...


class JsonParser(Parser):
    ...


p = JsonParser()
data = p.parse('{"key": "value"}')
print(JsonParser.supported_format(), data)
print(p.format({'x': 1}))

In [ ]:
# Ćw. 6
from abc import ABC, abstractmethod

class Validator(ABC):
    @abstractmethod
    def validate(self, value) -> bool:
        ...

    @abstractmethod
    def error_message(self) -> str:
        ...

    def check(self, value) -> None:
        ...


class RangeValidator(Validator):
    def __init__(self, min_val, max_val):
        ...


class LengthValidator(Validator):
    def __init__(self, min_len, max_len):
        ...


rv = RangeValidator(0, 00)
rv.check(50)   # OK
lv = LengthValidator(3, 10)
lv.check('hi')  # ValueError

---

## 3. 🔹 `isinstance()` / `issubclass()` z ABC

ABC pozwala rejestrować klasy jako 'wirtualne podklasy' bez
dziedziczenia - wystarczy wywołać `ABC.register(cls)`. Dzięki
temu `isinstance()` i `issubclass()` zwrócą `True`, nawet jeśli
klasa nie dziedziczy po ABC.

Przydaje się to przy integracji ze starszym kodem lub bibliotekami
zewnętrznymi, których nie możemy modyfikować.

| Sprawdzenie | Znaczenie |
|---|---|
| `isinstance(obj, ABC)` | czy obiekt jest instancją ABC lub podklasy |
| `issubclass(Cls, ABC)` | czy klasa jest podklasą ABC |
| `ABC.register(Cls)` | rejestracja wirtualnej podklasy |
| `__subclasshook__` | hook do customowego sprawdzania |

In [ ]:
from abc import ABC, abstractmethod


class Drawable(ABC):
    @abstractmethod
    def draw(self) -> None:
        ...

    @classmethod
    def __subclasshook__(cls, subclass):
        # każda klasa z metodą 'draw' to Drawable
        if cls is Drawable:
            return hasattr(subclass, 'draw')
        return NotImplemented


# Klasa zewnętrzna - nie dziedziczy po Drawable
class OldRenderer:
    def draw(self):
        print('OldRenderer.draw()')


print(issubclass(OldRenderer, Drawable))  # True - ma metodę draw
print(isinstance(OldRenderer(), Drawable))  # True


# Rejestracja bez dziedziczenia
class LegacyWidget:
    def draw(self):
        print('LegacyWidget.draw()')

Drawable.register(LegacyWidget)
print(isinstance(LegacyWidget(), Drawable))  # True

---

### 🐍 Ćwiczenia - isinstance/issubclass z ABC

**Ćw. 7.** Utwórz ABC `Iterable_` z `__subclasshook__`:
każda klasa z metodą `__iter__` jest uważana za jego podklasę.
Sprawdź `isinstance` dla `list`, `str`, `dict` i `int`.

**Ćw. 8.** Utwórz ABC `Printable` z metodą `pretty_print()`.
Zarejestruj w nim klasę `OldReport` (bez dziedziczenia).
Sprawdź `isinstance` przed i po rejestracji.

**Ćw. 9.** *(Trudniejsze)* Utwórz ABC `Plugin` z metodami
`name() -> str` i `execute() -> None`. Dodaj słownik
`REGISTRY: dict` do śledzenia zarejestrowanych pluginów.
Nadpisz `__init_subclass__` tak, by każda nieabstrakcyjna podklasa
automatycznie trafiała do `REGISTRY`.
# hint: sprawdź czy podklasa ma ustawione __abstractmethods__

In [ ]:
# Ćw. 7
from abc import ABC

class Iterable_(ABC):
    @classmethod
    def __subclasshook__(cls, subclass):
        ...


for t in [list, str, dict, int]:
    print(f'{t.__name__}: {issubclass(t, Iterable_)}')

In [ ]:
# Ćw. 8
from abc import ABC, abstractmethod

class Printable(ABC):
    @abstractmethod
    def pretty_print(self) -> None:
        ...


class OldReport:
    def pretty_print(self):
        print('OldReport')


print(isinstance(OldReport(), Printable))  # False
...
print(isinstance(OldReport(), Printable))  # True

In [ ]:
# Ćw. 9
from abc import ABC, abstractmethod

class Plugin(ABC):
    REGISTRY: dict = {}

    @abstractmethod
    def name(self) -> str:
        ...

    @abstractmethod
    def execute(self) -> None:
        ...

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        ...


class HelloPlugin(Plugin):
    def name(self) -> str: return 'hello'
    def execute(self) -> None: print('Hello, plugin!')


class GoodbyePlugin(Plugin):
    def name(self) -> str: return 'goodbye'
    def execute(self) -> None: print('Goodbye!')


print(Plugin.REGISTRY)
for plugin in Plugin.REGISTRY.values():
    plugin().execute()